# Capítulo 2 — Radiação solar e fotoperíodo

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/02_radiacao_solar.ipynb)

> Pré-requisito: Capítulo 1 (ambiente configurado, `agrometeorologiapy` instalada).

---


## 2.1 Motivação

A radiação solar é a fonte primária de energia para fotossíntese, aquecimento do solo e
evapotranspiração. Praticamente tudo que vem depois neste curso — evapotranspiração,
balanço hídrico, zoneamento — depende, direta ou indiretamente, de quanta energia chega
à superfície em cada dia e local.

Neste capítulo você vai calcular, a partir apenas de **latitude e data**, quanta radiação
o topo da atmosfera recebe (`Qo`), e a partir de temperatura ou de horas de brilho solar,
quanta radiação chega de fato à superfície (`Qg`). No final, vai fechar o balanço de energia
e converter energia disponível em lâmina de água evaporada — a ponte para o Capítulo 5
(evapotranspiração).


## 2.2 Grandezas fundamentais

- **Qo**: radiação solar extraterrestre (MJ m⁻² dia⁻¹) — quanto chegaria ao topo da atmosfera.
- **Qg**: radiação solar global na superfície (MJ m⁻² dia⁻¹) — o que efetivamente chega ao solo.
- **Rn**: saldo de radiação (MJ m⁻² dia⁻¹) — energia líquida disponível na superfície.
- **LE**: fluxo de calor latente — energia consumida pela evapotranspiração.
- **H**: fluxo de calor sensível — energia que aquece o ar.
- **G**: fluxo de calor no solo (≈ 0 em escala diária).

O balanço de energia fecha assim: `Rn = LE + H + G`. A **razão de Bowen** `β = H/LE` descreve
como essa energia se reparte entre aquecer o ar e evaporar água.


## 2.3 Fórmulas

**Declinação solar** $\delta$ (graus):
$$\delta = 23{,}45 \cdot \sin\left[\frac{360\,(284 + NDA)}{365}\right]$$

**Distância relativa Terra-Sol:**
$$\left(\frac{d}{D}\right)^2 = 1 + 0{,}033 \cdot \cos\left(NDA \times \frac{360}{365}\right)$$

**Ângulo horário do nascer do sol** $H_n$ (graus), $\varphi$ = latitude (negativa no hemisfério Sul):
$$\cos(H_n) = -\tan(\varphi)\cdot\tan(\delta)$$

**Radiação extraterrestre** $Q_o$ (MJ m⁻² dia⁻¹), com $H_n$ em radianos:
$$Q_o = 37{,}6 \cdot \left(\frac{d}{D}\right)^2 \left[H_n \sin(\varphi)\sin(\delta) + \cos(\varphi)\cos(\delta)\sin(H_n)\right]$$

**Radiação global — Angström-Prescott** (a partir de horas de brilho solar $n$ e fotoperíodo $N$):
$$Q_g = Q_o\left(a + b\,\frac{n}{N}\right),\quad a = 0{,}29\cos(\varphi),\ b = 0{,}52$$

**Radiação global — Hargreaves-Samani** (a partir só de $T_{max}$, $T_{min}$):
$$Q_g = k \cdot Q_o \cdot (T_{max} - T_{min})^{0{,}5},\quad k = 0{,}16\ \text{(continental)}$$

**Saldo de ondas curtas e saldo de radiação:**
$$BOC = Q_g\,(1-r),\ r = 0{,}25\ \text{(gramado)} \qquad R_n = BOC + BOL$$

**Razão de Bowen e partição de energia:**
$$\beta = \frac{H}{LE} \implies LE = \frac{R_n}{1+\beta},\quad H = \beta \cdot LE$$

**Conversão de LE para lâmina de água:**
$$\text{Lâmina (mm)} = \frac{LE\ (\text{MJ m}^{-2})}{\lambda},\quad \lambda = 2{,}45\ \text{MJ mm}^{-1}$$


## 2.4 Do papel ao código

A `agrometeorologiapy` já implementa todas as funções deste capítulo: `declinacao_solar`,
`angulo_horario_nascer`, `fator_correcao_distancia`, `irradiancia_extraterrestre` (Qo),
`Qg_hargreaves` (Qg), `boc_saldo`/`bol_saldo`/`saldo_radiacao` (Rn) — todas seguindo
exatamente as fórmulas acima. Vamos importar a biblioteca como `amp` e usar essas funções
diretamente; só implementamos aqui `particao_energia` e `lamina_evapotranspirada`, que não
têm equivalente pronto na biblioteca.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import agrometeorologiapy as amp


def particao_energia(Rn: float, beta: float) -> tuple[float, float]:
    """Retorna (LE, H) a partir do saldo de radiação e da razão de Bowen."""
    LE = Rn / (1 + beta)
    H = beta * LE
    return LE, H


def lamina_evapotranspirada(LE: float, lam: float = 2.45) -> float:
    """Converte fluxo de calor latente (MJ m-2 dia-1) em lâmina de água (mm dia-1)."""
    return LE / lam

## 2.5 Atividade guiada — reproduzindo o exercício de Cascavel-PR

Vamos reproduzir, em código, o exercício resolvido da apostila para **Cascavel-PR**
(φ = -25°, 15 de janeiro, NDA = 15): calcular Qo, Qg (Hargreaves-Samani), LE e a lâmina de
evapotranspiração, dados `Tmax = 33,0 °C`, `Tmin = 21,0 °C`, `Rn = 14,5 MJ m⁻² dia⁻¹`, `β = 0,3`.

Os resultados esperados (conforme a apostila): Qo ≈ 42,65, Qg ≈ 23,63, LE ≈ 11,15 e
lâmina ≈ 4,55 mm dia⁻¹.


In [ ]:
lat = -25
nda = 15
Tmax, Tmin = 33.0, 21.0
Rn = 14.5
beta = 0.3

declinacao = amp.declinacao_solar(nda)
Hn = amp.angulo_horario_nascer(lat, declinacao)
dD2 = amp.fator_correcao_distancia(nda)
Qo = amp.irradiancia_extraterrestre(lat, declinacao, Hn, dD2)
Qg = amp.Qg_hargreaves(Tmax, Tmin, Qo)
LE, H = particao_energia(Rn, beta)
lamina = lamina_evapotranspirada(LE)

print(f"Qo     = {Qo:.2f} MJ m-2 dia-1")
print(f"Qg     = {Qg:.2f} MJ m-2 dia-1")
print(f"LE     = {LE:.2f} MJ m-2 dia-1")
print(f"Lâmina = {lamina:.2f} mm dia-1")

Se os valores baterem com os da apostila, é sinal de que sabemos usar a `agrometeorologiapy`
corretamente — vamos reaproveitar essas mesmas funções, sem alteração, nos Capítulos 4 e 5.

## 2.6 Aplicando em dados reais

Reaproveitando o `df_clima` de Santa Helena-PR baixado no Capítulo 1 (NASA POWER), vamos
calcular `Qo` para cada dia do ano e comparar com `ALLSKY_SFC_SW_DWN` (radiação global
medida por satélite, nosso `Qg` "observado"). Se você não rodou o Capítulo 1 nesta sessão,
execute a célula de download abaixo primeiro.


In [ ]:
import requests

LAT, LON = -24.86, -54.33  # Santa Helena-PR
url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
    "community": "AG",
    "longitude": LON,
    "latitude": LAT,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}
resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()
propriedades = resposta.json()["properties"]["parameter"]

df_clima = pd.DataFrame(propriedades)
df_clima.index = pd.to_datetime(df_clima.index, format="%Y%m%d")
df_clima.index.name = "data"
df_clima = df_clima.replace(-999, np.nan)
df_clima.head(2)


In [ ]:
def calcular_Qo(lat, n):
    declinacao = amp.declinacao_solar(n)
    Hn = amp.angulo_horario_nascer(lat, declinacao)
    dD2 = amp.fator_correcao_distancia(n)
    return amp.irradiancia_extraterrestre(lat, declinacao, Hn, dD2)


df_clima["nda"] = df_clima.index.dayofyear
df_clima["Qo"] = df_clima["nda"].apply(lambda n: calcular_Qo(LAT, n))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_clima.index, df_clima["Qo"], label="Qo (extraterrestre, calculado)", color="orange")
ax.plot(df_clima.index, df_clima["ALLSKY_SFC_SW_DWN"], label="Qg (superfície, NASA POWER)", color="steelblue")
ax.set_ylabel("MJ m-2 dia-1")
ax.set_title("Santa Helena-PR — Qo x Qg observado (2023)")
ax.legend()
plt.tight_layout()
plt.show()

Repare que `Qg` (observado) é sempre menor que `Qo` (extraterrestre) — a diferença é o que
é absorvido/refletido pela atmosfera (nuvens, vapor d'água, aerossóis). É exatamente essa
diferença que a fórmula de Hargreaves-Samani tenta estimar a partir da amplitude térmica.


## 2.7 Desafio

1. Calcule e plote `Qo` ao longo de um ano completo (365 dias) para três latitudes:
   -25° (Paraná), 0° (equador) e -60° (próximo à Antártida). Compare as curvas e explique
   a diferença de amplitude sazonal entre elas.
2. Compare, para Santa Helena-PR, o `Qg` estimado por Hargreaves-Samani (usando `T2M_MAX` e
   `T2M_MIN` do `df_clima`) com o `Qg` observado (`ALLSKY_SFC_SW_DWN`). Calcule o erro médio
   absoluto (MAE) entre as duas séries.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 2.8 Checkpoint

Antes de seguir para o **Capítulo 3 — Temperatura, graus-dia e fenologia**, você deve ter:

- [ ] reproduzido o exercício de Cascavel-PR com os mesmos resultados da apostila;
- [ ] usado `amp.irradiancia_extraterrestre(lat, declinacao, Hn, dD2)` para calcular Qo em
      qualquer local/data;
- [ ] comparado Qo calculado com Qg observado em dados reais;
- [ ] resolvido pelo menos o item 1 do desafio.